# Jude 2026a · touch typing T17 and T18

## Prepare the data

Run the commands below from the repository root with `bci-repro` active.

### 1. Download

Download `Data.zip` from [Dryad](https://doi.org/10.5061/dryad.cz8w9gjjk). Create `studies/20260316_jude/data/raw/dryad_cz8w9gjjk/` and save the download as `studies/20260316_jude/data/raw/dryad_cz8w9gjjk/dryad_cz8w9gjjk__Data.zip`.

### 2. Extract

Extract the archive into `studies/20260316_jude/data/extracted/dryad_cz8w9gjjk/`.

```text
python -m zipfile -e "studies/20260316_jude/data/raw/dryad_cz8w9gjjk/dryad_cz8w9gjjk__Data.zip" "studies/20260316_jude/data/extracted/dryad_cz8w9gjjk"
```

After extraction, check for `studies/20260316_jude/data/extracted/dryad_cz8w9gjjk/Data/t17/`. The extracted `Data/` directory also contains `t18/`.

### 3. Prepare the model input

Convert the session MAT files to 20 ms TFRecords, then export an NPZ for PyTorch. This example uses T17's April 25 session, blocks 1, 3, 4, and 5, with 20 test trials.

```text
python shared/tools/setup_upstream.py --study 20260316_jude
python studies/20260316_jude/code/prepare_data.py t17.2024.04.25 1,3,4,5 20
python studies/20260316_jude/code/pytorch_port/export_tfrecord_to_npz.py t17.2024.04.25
```

**TFRecord output** `studies/20260316_jude/data/extracted/dryad_cz8w9gjjk/Data/t17/t17.2024.04.25/Typing/tfdata_20ms/`.

**NPZ output** `studies/20260316_jude/data/extracted/dryad_cz8w9gjjk/Data/t17/t17.2024.04.25/Typing/m1_torch_data.npz`.

## Read the archived results

The released test trial split was used for checkpoint validation. Figure 3b applies a minimum of three trials per session. Raw greedy CER is available at session level. The per-trial character and word counts below describe the post-LM output.

This walkthrough uses the checked-in archive and takes no data download.

In [1]:
import csv
import json
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "figures/code"))
from figure_paths import PUBLISHED, load_benchmark

benchmark = load_benchmark()

def read_rows(path):
    """Read the archived CSV rows in their original order."""
    with (root / path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

## From a trial or day to the reported result

In [2]:
from build_session_cer_vs_wer import build_jude26a
source = benchmark["figure3_cer_sources"]["Jude 2026a (T17)"]
archive = json.loads((root / source["archive"]).read_text(encoding="utf-8"))
day, data = next(iter(archive["t17"].items()))
trial = data["trials"][0]
print(trial["ref"], "→", trial["hyp"])
print(f"{day}: trial word errors {trial['we']}/{trial['nw']}")
for participant in ("t17", "t18"):
    result = build_jude26a(participant)
    print(f"{participant}: {len(result['sessions'])} sessions, {sum(s['n'] for s in result['sessions'])} eligible trials; pooled {result['pooled']}")

how very right → how very right
t17.2024.04.25: trial word errors 0/3
t17: 8 sessions, 134 eligible trials; pooled {'cer': 21.52, 'wer': 14.65}
t18: 7 sessions, 140 eligible trials; pooled {'cer': 8.41, 'wer': 7.63}


## Continue the full reproduction

If you use the 5-gram decoder, download `LanguageModels.zip` and save it at `studies/20260316_jude/data/raw/dryad_cz8w9gjjk/dryad_cz8w9gjjk__LanguageModels.zip`. Extract it under `studies/20260316_jude/data/extracted/dryad_cz8w9gjjk/`, then move the extracted `studies/20260316_jude/data/extracted/dryad_cz8w9gjjk/LanguageModels/Typing5gramPunc/` folder to `studies/20260316_jude/data/language_models/wfst_5gram/`.

For a separate data disk, set `BCI_DATA_ROOT` to the directory that contains the study folders and use the same layout under that root.

See the [evaluation and input guide](docs/20261124_evaluation_protocol/README.md) and [code guide](code/README.md).

**Public commands** [`prepare_data.py`](code/prepare_data.py), [`train.py`](code/train.py), [`infer.py`](code/infer.py), [`decode.py`](code/decode.py), [`evaluate.py`](code/evaluate.py). Run a command with `--help` to see its arguments.